## Frame Extracting

In [1]:
import cv2
import os


input_video_path = r"D:\Downloads\IP_Project\Road_Damaged_Video.mp4"


extract_frame_dir = r"D:\Downloads\IP_Project\Extracted_frames"


if not os.path.exists(extract_frame_dir):
    os.makedirs(extract_frame_dir)


cap = cv2.VideoCapture(input_video_path)

frame_count = 0


while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_name = os.path.join(
        extract_frame_dir,
        f"frame_{frame_count:05d}.jpg"
    )

    
    cv2.imwrite(frame_name, frame)

    frame_count += 1


cap.release()

print(f"Extracted {frame_count} frames successfully.")

Extracted 453 frames successfully.


## Enhance Pipeline


In [2]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

input_dir = r"D:\Downloads\IP_Project\Extracted_frames"
output_dir = r"D:\Downloads\IP_Project\Enhanced_frames"
histogram_dir = r"D:\Downloads\IP_Project\Enhancement_Histograms"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(histogram_dir, exist_ok=True)

# Noise threshold using standard deviation
NOISE_STD_THRESHOLD = 45

def gamma_correction(img, gamma):
    inv_gamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** inv_gamma * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, table)

def enhance_frame(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # -------------------------------
    # Noise recognition using std
    # -------------------------------
    noise_std = np.std(l)

    if noise_std > NOISE_STD_THRESHOLD:
        print("Noise detected | STD:", round(noise_std, 2))
        l = cv2.medianBlur(l, 3)
    else:
        print("Low noise | STD:", round(noise_std, 2))

    # -------------------------------
    # Contrast enhancement
    # -------------------------------
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    l = clahe.apply(l)

    # -------------------------------
    # Brightness correction
    # -------------------------------
    mean_val = np.mean(l)

    if mean_val < 100:
        l = gamma_correction(l, 1.2)
    elif mean_val > 170:
        l = gamma_correction(l, 0.8)

    lab = cv2.merge((l, a, b))
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # -------------------------------
    # Edge-preserving smoothing
    # -------------------------------
    smooth = cv2.bilateralFilter(enhanced, 5, 40, 40)

    # -------------------------------
    # Sharpening
    # -------------------------------
    blur = cv2.GaussianBlur(smooth, (0, 0), 1.0)
    sharp = cv2.addWeighted(smooth, 1.45, blur, -0.45, 0)

    return sharp, noise_std

files = sorted(
    glob.glob(os.path.join(input_dir, "*.jpg")) +
    glob.glob(os.path.join(input_dir, "*.png")) +
    glob.glob(os.path.join(input_dir, "*.jpeg"))
)

print("Total frames found:", len(files))

for i, path in enumerate(files, start=1):
    img = cv2.imread(path)

    if img is None:
        continue

    result, noise_std = enhance_frame(img)

    frame_name = os.path.basename(path)

    save_path = os.path.join(output_dir, f"enhanced_frame_{i:03d}.jpg")
    cv2.imwrite(save_path, result)

    if i <= 10:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        enhanced_gray = cv2.cvtColor(result, cv2.COLOR_BGR2GRAY)

        plt.figure(figsize=(14, 7))

        plt.subplot(2, 2, 1)
        plt.imshow(gray, cmap="gray")
        plt.title(f"Original Gray: {frame_name}\nNoise STD: {noise_std:.2f}")
        plt.axis("off")

        plt.subplot(2, 2, 2)
        plt.imshow(enhanced_gray, cmap="gray")
        plt.title("Enhanced Result")
        plt.axis("off")

        plt.subplot(2, 2, 3)
        plt.hist(gray.ravel(), bins=256, range=(0, 256))
        plt.title("Original Histogram")

        plt.subplot(2, 2, 4)
        plt.hist(enhanced_gray.ravel(), bins=256, range=(0, 256))
        plt.title("Enhanced Histogram")

        plt.tight_layout()

        hist_save_path = os.path.join(histogram_dir, f"histogram_compare_{i:03d}.jpg")
        plt.savefig(hist_save_path)
        plt.close()

print("Enhanced frames completed.")
print("Histogram comparisons saved in:", histogram_dir)

Total frames found: 453
Low noise | STD: 16.89
Low noise | STD: 16.61
Low noise | STD: 16.58
Low noise | STD: 16.53
Low noise | STD: 16.75
Low noise | STD: 16.62
Low noise | STD: 16.64
Low noise | STD: 16.59
Low noise | STD: 16.84
Low noise | STD: 16.79
Low noise | STD: 16.88
Low noise | STD: 16.89
Low noise | STD: 17.2
Low noise | STD: 17.16
Low noise | STD: 17.33
Low noise | STD: 17.48
Low noise | STD: 17.78
Low noise | STD: 17.93
Low noise | STD: 18.07
Low noise | STD: 18.09
Low noise | STD: 18.28
Low noise | STD: 18.09
Low noise | STD: 18.04
Low noise | STD: 17.98
Low noise | STD: 18.1
Low noise | STD: 18.0
Low noise | STD: 17.97
Low noise | STD: 17.84
Low noise | STD: 17.85
Low noise | STD: 17.74
Low noise | STD: 17.71
Low noise | STD: 17.7
Low noise | STD: 17.88
Low noise | STD: 17.85
Low noise | STD: 17.9
Low noise | STD: 17.9
Low noise | STD: 18.06
Low noise | STD: 18.05
Low noise | STD: 18.14
Low noise | STD: 18.22
Low noise | STD: 18.54
Low noise | STD: 18.5
Low noise | STD: 